# Cài đặt thư viện

In [ ]:
%pip install yfinance pandas numpy

# Tải và lưu dữ liệu


In [2]:
import requests
from bs4 import BeautifulSoup
import csv

def scrape_bitcoin_historical_data():
    # 1. Khai báo URL mục tiêu
    url = "https://finance.yahoo.com/quote/BTC-USD/history/?period1=1410912000&period2=1790400923"
    
    # BẮT BUỘC: Giả lập User-Agent để server tưởng đây là trình duyệt thật (tránh bị chặn HTTP 403)
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    print("1. Đang gửi yêu cầu HTTP đến trang web...")
    response = requests.get(url, headers=headers)

    # Kiểm tra trạng thái kết nối
    if response.status_code != 200:
        print(f"❌ Lỗi kết nối! Mã lỗi: {response.status_code}")
        return

    print("2. Tải trang thành công. Đang phân tích cấu trúc HTML...")
    # 2. Đưa toàn bộ mã nguồn HTML vào BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')

    # 3. Tìm thẻ <table> chứa dữ liệu lịch sử
    # Có thể dùng soup.find('table', class_='tên-class-nếu-có') để chỉ định chính xác hơn
    table = soup.find('table')

    if not table:
        print("❌ Không tìm thấy thẻ <table>. Web có thể đã dùng JavaScript để render bảng này.")
        return

    # 4. Tạo và mở file CSV để ghi dữ liệu
    filename = 'btc_scraped_data.csv'
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        
        # Ghi dòng tiêu đề (Header)
        writer.writerow(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close'])
        
        # 5. Xác định phần thân bảng và duyệt qua từng hàng
        tbody = table.find('tbody')
        rows = tbody.find_all('tr')
        
        print(f"3. Tìm thấy {len(rows)} dòng dữ liệu. Đang tiến hành bóc tách...")
        
        valid_rows_count = 0
        for row in rows:
            # Lấy tất cả các ô dữ liệu <td> trong hàng hiện tại
            cols = row.find_all('td')
            
            # Một dòng dữ liệu giá hợp lệ thường có đúng 7 cột
            if len(cols) == 7:
                # Trích xuất text bên trong thẻ <td> và loại bỏ khoảng trắng thừa
                date = cols[0].text.strip()
                open_price = cols[1].text.strip()
                high = cols[2].text.strip()
                low = cols[3].text.strip()
                close = cols[4].text.strip()
                adj_close = cols[5].text.strip()
                
                # Ghi mảng dữ liệu vào file CSV
                writer.writerow([date, open_price, high, low, close, adj_close])
                valid_rows_count += 1
                
        print(f"✅ Xong! Đã cào và lưu thành công {valid_rows_count} dòng vào file '{filename}'.")

# Khởi chạy hàm
scrape_bitcoin_historical_data()

1. Đang gửi yêu cầu HTTP đến trang web...
2. Tải trang thành công. Đang phân tích cấu trúc HTML...
3. Tìm thấy 4392 dòng dữ liệu. Đang tiến hành bóc tách...
✅ Xong! Đã cào và lưu thành công 4392 dòng vào file 'btc_scraped_data.csv'.


# Tải toàn bộ dữ liệu S&P 500 (bắt đầu từ 30/12/1927)


In [ ]:
import yfinance as yf
import pandas as pd

# S&P 500 Index
ticker = "^GSPC"

# 1. Download dữ liệu S&P 500 bắt đầu từ ngày 30/12/1927
df = yf.download(
    ticker,
    start="1927-12-30",
    interval="1d",
    auto_adjust=False,
    actions=False,
    progress=True
)

# 2. Xử lý MultiIndex nếu yfinance trả về bảng 2 tầng
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Xóa tên index cột nếu có (bỏ header 'Price')
df.columns.name = None

# 3. Đưa Date thành column và format chuẩn YYYY-MM-DD
df = df.reset_index()
df["Date"] = pd.to_datetime(df["Date"]).dt.strftime("%Y-%m-%d")

# Lọc chính xác từ ngày 30/12/1927
df = df[df["Date"] >= "1927-12-30"]

# 4. Sắp xếp thứ tự các cột chuẩn bao gồm cả Volume
desired_cols = ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]
df = df[[c for c in desired_cols if c in df.columns]]

# 5. Kiểm tra dữ liệu
print("Dòng đầu tiên:\n", df.head())
print("\nDòng cuối cùng:\n", df.tail())
print("\nKích thước dữ liệu:", df.shape)
print("Danh sách các cột:", df.columns.tolist())
print("Ngày đầu tiên:", df.iloc[0]["Date"])

# 6. Lưu file CSV
output_file = "sp500_yahoo.csv"
df.to_csv(output_file, index=False)
print(f"\n✅ Đã lưu thành công: {output_file}")

# Cào dữ liệu S&P 500 bằng Selenium và lưu 3 định dạng: CSV, TSV, JSON


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from io import StringIO
import pandas as pd
import time


def scrape_sp500():
    options = webdriver.ChromeOptions()
    options.add_argument("--window-size=1920,1080")
    driver = webdriver.Chrome(options=options)

    try:
        url = (
            "https://finance.yahoo.com/quote/%5EGSPC/history/"
            "?period1=-1325583000&period2=1790689249&frequency=1d"
        )
        driver.get(url)

        table = WebDriverWait(driver, 30).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "table"))
        )

        table_html = table.get_attribute("outerHTML")

        df = pd.read_html(StringIO(table_html))[0]

        df = df[
            ~df.astype(str).apply(
                lambda row: row.str.contains(r"\*", regex=True).any(),
                axis=1
            )
        ]

        df.columns = [
            "Date",
            "Open",
            "High",
            "Low",
            "Close",
            "Adj Close",
            "Volume"
        ]

        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.dropna(subset=["Date"])
        df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")

        df.to_csv("sp500_yahoo.csv", index=False, encoding="utf-8")
        df.to_csv("sp500_yahoo.tsv", sep="\t", index=False, encoding="utf-8")
        df.to_json("sp500_yahoo.json", orient="records", indent=4, force_ascii=False)

        return df

    except Exception:
        pass

    finally:
        driver.quit()


df = scrape_sp500()